# P03 — Differential Privacy con PyTorch y Opacus

Velora Finance quiere entrenar un modelo de riesgo reduciendo la influencia observable de cada registro individual en el modelo publicado.

La práctica compara:

1. un modelo PyTorch convencional;
2. el mismo tipo de modelo entrenado con **DP-SGD mediante Opacus**.

Además analizaremos **qué parámetros construyen realmente la garantía** y cómo se acumula el presupuesto de privacidad a lo largo del ciclo de vida.


## 1. Activar GPU T4

En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

La GPU se utiliza para el entrenamiento PyTorch/Opacus; no constituye por sí misma ningún mecanismo de privacidad.


In [ ]:
!pip -q install opacus==1.6.0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score

from opacus import PrivacyEngine

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Dataset

Generamos 30.000 solicitudes con el mismo proceso sintético de Velora.

Un conjunto mayor permite que el experimento DP-SGD sea más representativo y hace visible la relación entre tamaño de dataset, sampling y presupuesto de privacidad.


In [ ]:
GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"

namespace = {"__name__": "velora_generator"}
with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)

df = namespace["generate_dataset"](n=30000, seed=42)

print(df.shape)
print(f"Tasa de impago: {df['default'].mean():.2%}")


## 3. Preparar train/test


In [ ]:
FEATURES = [
    "age", "annual_income", "debt_ratio", "employment_months",
    "late_payments_12m", "credit_history_months", "existing_debt",
    "requested_amount", "savings_balance",
]

X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES],
    df["default"],
    test_size=0.20,
    random_state=42,
    stratify=df["default"],
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

y_train_np = y_train.to_numpy(dtype="int64")
y_test_np = y_test.to_numpy(dtype="int64")

train_ds = TensorDataset(
    torch.tensor(X_train_scaled),
    torch.tensor(y_train_np),
)

test_x = torch.tensor(X_test_scaled, device=device)

print("Train:", len(train_ds))
print("Test :", len(y_test_np))


## 4. Modelo

Usamos una red pequeña. El objetivo no es maximizar rendimiento, sino comparar de forma controlada entrenamiento convencional y entrenamiento privado.


In [ ]:
def make_model():
    return nn.Sequential(
        nn.Linear(len(FEATURES), 32),
        nn.ReLU(),
        nn.Linear(32, 16),
        nn.ReLU(),
        nn.Linear(16, 2),
    )

def evaluate(model):
    model.eval()

    with torch.no_grad():
        logits = model(test_x)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        pred = (probs >= 0.5).astype(int)

    return {
        "roc_auc": roc_auc_score(y_test_np, probs),
        "accuracy": accuracy_score(y_test_np, pred),
        "recall": recall_score(y_test_np, pred, zero_division=0),
    }

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)

    return total_loss / len(loader.dataset)


## 5. Entrenamiento convencional

Primero obtenemos una referencia sin Differential Privacy.


In [ ]:
BATCH_SIZE = 512
EPOCHS = 8

torch.manual_seed(42)

baseline_model = make_model().to(device)
baseline_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

baseline_optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=1e-3,
)

criterion = nn.CrossEntropyLoss()

baseline_history = []

for epoch in range(EPOCHS):
    loss = train_epoch(
        baseline_model,
        baseline_loader,
        baseline_optimizer,
        criterion,
    )
    baseline_history.append({
        "epoch": epoch + 1,
        "loss": loss,
    })

baseline_metrics = evaluate(baseline_model)

print("Modelo convencional")
for k, v in baseline_metrics.items():
    print(f"{k:10}: {v:.3f}")


# 6. Entrenamiento privado con Opacus

DP-SGD modifica el entrenamiento en tres puntos fundamentales:

1. calcula gradientes por ejemplo;
2. aplica **clipping** para limitar la sensibilidad;
3. añade ruido antes de actualizar el modelo.

El `PrivacyEngine` además contabiliza el presupuesto acumulado.


In [ ]:
TARGET_EPSILON = 3.0 #@param {type:"slider", min:0.5, max:10.0, step:0.5}
TARGET_DELTA = 1e-5
MAX_GRAD_NORM = 1.0

torch.manual_seed(42)

private_model = make_model().to(device)

private_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

private_optimizer = torch.optim.Adam(
    private_model.parameters(),
    lr=1e-3,
)

privacy_engine = PrivacyEngine(accountant="prv")

private_model, private_optimizer, private_loader = (
    privacy_engine.make_private_with_epsilon(
        module=private_model,
        optimizer=private_optimizer,
        data_loader=private_loader,
        target_epsilon=TARGET_EPSILON,
        target_delta=TARGET_DELTA,
        epochs=EPOCHS,
        max_grad_norm=MAX_GRAD_NORM,
    )
)

print("Entrenamiento privado preparado.")


## 7. ¿Qué configuración produce la garantía?

`epsilon` no aparece por arte de magia.

Opacus calcula el ruido necesario teniendo en cuenta, entre otros factores:

- objetivo de `epsilon`;
- `delta`;
- número de epochs;
- tasa de muestreo;
- clipping;
- mecanismo de contabilidad.

Vamos a inspeccionar los parámetros efectivos.


In [ ]:
noise_multiplier = private_optimizer.noise_multiplier
max_grad_norm = private_optimizer.max_grad_norm
sample_rate = 1 / len(private_loader)

privacy_configuration = pd.Series({
    "target_epsilon": TARGET_EPSILON,
    "target_delta": TARGET_DELTA,
    "epochs": EPOCHS,
    "batch_size_nominal": BATCH_SIZE,
    "sample_rate_aprox": sample_rate,
    "noise_multiplier": noise_multiplier,
    "max_grad_norm": max_grad_norm,
})

display(privacy_configuration.to_frame("valor"))


### Cómo interpretar los parámetros

- `max_grad_norm` limita cuánto puede influir el gradiente de un ejemplo.
- `noise_multiplier` expresa cuánto ruido gaussiano se añade en relación con la sensibilidad acotada.
- `sample_rate` y el número de pasos afectan a la contabilidad.
- `(ε, δ)` resume la garantía resultante para la ejecución.

Una afirmación como **“el modelo tiene ε = 3”** sin indicar el resto del mecanismo es incompleta.


## 8. Observar cómo se consume el presupuesto

Registramos `epsilon` después de cada epoch.


In [ ]:
private_history = []

for epoch in range(EPOCHS):
    loss = train_epoch(
        private_model,
        private_loader,
        private_optimizer,
        criterion,
    )

    epsilon_now = privacy_engine.get_epsilon(TARGET_DELTA)

    private_history.append({
        "epoch": epoch + 1,
        "loss": loss,
        "epsilon": epsilon_now,
    })

history_df = pd.DataFrame(private_history)
display(history_df.style.format({
    "loss": "{:.4f}",
    "epsilon": "{:.3f}",
}))

plt.plot(
    history_df["epoch"],
    history_df["epsilon"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Epsilon acumulado")
plt.title("Consumo del presupuesto durante entrenamiento")
plt.show()


### Idea clave

El presupuesto de privacidad **se consume durante el entrenamiento**.

Más pasos de optimización pueden mejorar utilidad, pero también afectan al presupuesto. Por eso epochs, sampling, clipping, ruido y objetivo de utilidad forman una decisión conjunta.


## 9. Resultado privado y comparación de utilidad


In [ ]:
private_metrics = evaluate(private_model)
spent_epsilon = privacy_engine.get_epsilon(TARGET_DELTA)

print("Modelo con DP-SGD")
print(f"epsilon gastado: {spent_epsilon:.3f}")
print(f"delta          : {TARGET_DELTA}")

for k, v in private_metrics.items():
    print(f"{k:10}: {v:.3f}")

comparison = pd.DataFrame([
    {"modelo": "Sin DP", **baseline_metrics},
    {"modelo": f"DP-SGD ε≈{spent_epsilon:.2f}", **private_metrics},
]).set_index("modelo")

display(comparison.style.format("{:.3f}"))


### Qué significa `(ε, δ)`

La garantía no dice “nadie puede reconstruir ningún dato”.

Formaliza que la distribución de resultados del mecanismo cambia de forma limitada cuando un único ejemplo entra o sale del conjunto de entrenamiento.

`epsilon` debe interpretarse junto con `delta`, el mecanismo, el dataset, la tasa de muestreo, el número de pasos y el modelo de amenazas.


## 10. Cambiar el presupuesto objetivo

Ejecuta de nuevo la parte privada con, por ejemplo:

- `ε = 1`
- `ε = 3`
- `ε = 8`

Compara:

- `noise_multiplier`;
- ROC-AUC;
- Recall;
- epsilon gastado.

No buscamos “el epsilon ganador”. Buscamos comprender qué garantía estamos comprando y a qué coste.


## 11. Conexión con fairness

Una técnica de privacidad puede cambiar el rendimiento de forma diferente entre subgrupos.

Comparamos Recall y ROC-AUC por `customer_group`.


In [ ]:
test_groups = df.loc[X_test.index, "customer_group"].to_numpy()

def probabilities(model):
    model.eval()
    with torch.no_grad():
        return torch.softmax(
            model(test_x),
            dim=1
        )[:, 1].cpu().numpy()

base_probs = probabilities(baseline_model)
priv_probs = probabilities(private_model)

rows = []

for group in sorted(np.unique(test_groups)):
    mask = test_groups == group

    for name, probs in [
        ("Sin DP", base_probs),
        ("DP-SGD", priv_probs),
    ]:
        pred = (probs[mask] >= 0.5).astype(int)

        rows.append({
            "grupo": group,
            "modelo": name,
            "recall_impago": recall_score(
                y_test_np[mask],
                pred,
                zero_division=0,
            ),
            "roc_auc": roc_auc_score(
                y_test_np[mask],
                probs[mask],
            ),
        })

group_utility = pd.DataFrame(rows)

display(
    group_utility
    .pivot(
        index="grupo",
        columns="modelo",
        values=["recall_impago", "roc_auc"],
    )
    .style.format("{:.3f}")
)


### Interpretación

Una garantía de privacidad no garantiza que la pérdida de utilidad se reparta de forma uniforme.

Esto obliga a evaluar conjuntamente:

**privacy ↔ utility ↔ fairness**


# 12. Composición: privacidad durante el ciclo de vida

Supongamos que Velora no entrena una sola vez.

Reentrena periódicamente sobre poblaciones que pueden solaparse.

Como aproximación conservadora de **composición secuencial básica**:

\[
\epsilon_{total} \leq \sum_i \epsilon_i
\]

\[
\delta_{total} \leq \sum_i \delta_i
\]

La contabilidad real puede utilizar métodos más ajustados, pero esta cota sirve para entender el problema de gobierno.


In [ ]:
EPSILON_POR_ENTRENAMIENTO = 3.0 #@param {type:"slider", min:0.5, max:10.0, step:0.5}
NUM_REENTRENAMIENTOS = 4 #@param {type:"slider", min:1, max:12, step:1}
DELTA_POR_ENTRENAMIENTO = 1e-5

epsilon_basic = (
    EPSILON_POR_ENTRENAMIENTO
    * NUM_REENTRENAMIENTOS
)

delta_basic = (
    DELTA_POR_ENTRENAMIENTO
    * NUM_REENTRENAMIENTOS
)

print("Cota de composición secuencial básica")
print(f"ε_total <= {epsilon_basic:.2f}")
print(f"δ_total <= {delta_basic:.2e}")


### Tres escenarios distintos

**A. Mismos usuarios o población parcialmente solapada**  
La composición acumulada importa.

**B. Conjuntos realmente disjuntos**  
La composición puede comportarse de forma diferente; no debemos sumar automáticamente sin analizar el mecanismo y la población.

**C. Consultas, reentrenamientos y modelos posteriores**  
El presupuesto debe gestionarse a nivel de sistema, no solo dentro de un notebook.

La pregunta de gobierno es:

> ¿Quién autoriza gastar más presupuesto de privacidad y cómo se registra?


## 13. Federated Learning: complementario, no sustituto

Federated Learning reduce la necesidad de centralizar los datos brutos: el entrenamiento se ejecuta en clientes o silos y se intercambian actualizaciones.

Pero FL **no proporciona por sí solo Differential Privacy**.

Siguen existiendo riesgos como:

- inferencia a partir de actualizaciones;
- poisoning;
- clientes maliciosos;
- metadatos;
- compromiso del agregador.

Una implementación real con Flower introduce además arquitectura de federación (`ClientApp`, `ServerApp`, estrategia, simulación/despliegue). No la añadimos a P03 porque aquí el objetivo es profundizar en la garantía de privacidad, no en operar un framework de FL.


# Conclusiones de P03

- DP-SGD aplica clipping, ruido y contabilidad durante un entrenamiento real.
- `epsilon` no es un parámetro aislado: depende del mecanismo completo y de cómo se utiliza.
- El presupuesto se consume a medida que avanza el entrenamiento.
- Una garantía de privacidad tiene un coste de utilidad medible.
- Ese coste puede distribuirse de manera diferente entre grupos.
- El presupuesto debe gestionarse a lo largo del ciclo de vida y de los reentrenamientos.
- Federated Learning y Differential Privacy son complementarios, pero resuelven problemas diferentes.
- La pregunta correcta no es “¿usamos DP?”, sino **qué amenaza reducimos, qué garantía obtenemos, qué utilidad perdemos y cómo gobernamos esa decisión**.
